In [1]:
import numpy as np
import torch
from dld.models.get_model import get_module
from dld.data.get_data import get_datasets
from omegaconf import OmegaConf

In [2]:
cfg_local_path = "/home/LODGE/exp/Local_Module/FineDance_FineTuneV2_Local/local_train.yaml"
cfg_assets_path = "/home/LODGE/configs/data/assets.yaml"
cfg_local = OmegaConf.load(cfg_local_path)
cfg_assets = OmegaConf.load(cfg_assets_path)
cfg = OmegaConf.merge(cfg_local, cfg_assets)
cfg.checkpoint1 = 'exp/Global_Module/FineDance_Global/checkpoints/epoch=2999.ckpt'
cfg.checkpoint2 = 'exp/Local_Module/FineDance_FineTuneV2_Local/checkpoints/epoch=299.ckpt'

In [6]:
import os
import glob

def rename_files_with_padding(directory):
    files = sorted(os.listdir(directory))
    for filename in files:
        name, ext = os.path.splitext(filename)
        if name.isdigit():
            new_name = f"{int(name):03d}{ext}"
            old_path = os.path.join(directory, filename)
            new_path = os.path.join(directory, new_name)
            os.rename(old_path, new_path)
            print(f"Renamed: {filename} -> {new_name}")

parent_folder_path = "/home/LODGE/data/finedance-kpop"
folder_paths = [path for path in glob.glob(os.path.join(parent_folder_path, "*")) if os.path.isdir(path)]
for folder_path in folder_paths:
    rename_files_with_padding(folder_path)
    print(f"Renamed files in: {folder_path}")


Renamed: 0.npy -> 000.npy
Renamed: 1.npy -> 001.npy
Renamed: 10.npy -> 010.npy
Renamed: 100.npy -> 100.npy
Renamed: 101.npy -> 101.npy
Renamed: 102.npy -> 102.npy
Renamed: 103.npy -> 103.npy
Renamed: 104.npy -> 104.npy
Renamed: 105.npy -> 105.npy
Renamed: 106.npy -> 106.npy
Renamed: 107.npy -> 107.npy
Renamed: 108.npy -> 108.npy
Renamed: 109.npy -> 109.npy
Renamed: 11.npy -> 011.npy
Renamed: 110.npy -> 110.npy
Renamed: 111.npy -> 111.npy
Renamed: 112.npy -> 112.npy
Renamed: 113.npy -> 113.npy
Renamed: 114.npy -> 114.npy
Renamed: 115.npy -> 115.npy
Renamed: 116.npy -> 116.npy
Renamed: 117.npy -> 117.npy
Renamed: 118.npy -> 118.npy
Renamed: 119.npy -> 119.npy
Renamed: 12.npy -> 012.npy
Renamed: 120.npy -> 120.npy
Renamed: 121.npy -> 121.npy
Renamed: 122.npy -> 122.npy
Renamed: 123.npy -> 123.npy
Renamed: 124.npy -> 124.npy
Renamed: 125.npy -> 125.npy
Renamed: 126.npy -> 126.npy
Renamed: 127.npy -> 127.npy
Renamed: 128.npy -> 128.npy
Renamed: 129.npy -> 129.npy
Renamed: 13.npy -> 013.npy


In [3]:
dataset = get_datasets(cfg)

In [4]:
model = get_module(cfg, dataset)
model

Local_Module(
  (DanceDecoder): Refine_DanceDecoder(
    (mapping): MappingNet(
      (shared): Sequential(
        (0): Linear(in_features=256, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=512, out_features=512, bias=True)
        (3): GELU(approximate='none')
      )
      (unshared): ModuleList(
        (0): Sequential(
          (0): Linear(in_features=512, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=512, bias=True)
        )
        (1): Sequential(
          (0): Linear(in_features=512, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=512, bias=True)
        )
        (2): Sequential(
          (0): Linear(in_features=512, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=512, bias=True)
        )
        (3): 

In [5]:
state_dict = torch.load(cfg.checkpoint2, map_location="cpu")["state_dict"]

In [7]:
model.load_state_dict(state_dict, strict=True)

In [8]:
import torch.nn as nn
from peft import LoraConfig, TaskType
from dld.models.architectures.lora import LoRA_MappingNet, freeze_all_except_lora

NEW_GENRE_ID = 16

# === Define LoRA config ===
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    # target_modules=["0", "2"],
    # task_type=TaskType.FEATURE_EXTRACTION,
)

# === Replace genre mappings with LoRA versions ===
def wrap_mapping_with_lora(mappingnet, in_dim, dim, new_genre_id, lora_cfg):
    base_class = lambda: nn.Sequential(*[layer for layer in mappingnet.unshared[0]])

    wrapped = LoRA_MappingNet(
        in_dim=in_dim,
        dim=dim,
        genre_num=new_genre_id + 1,
        base_class=base_class,
        lora_cfg=lora_cfg,
        new_genre_id=new_genre_id,
    )

    wrapped.shared.load_state_dict(mappingnet.shared.state_dict())

    for i in range(new_genre_id):
        wrapped.unshared[str(i)].load_state_dict(mappingnet.unshared[i].state_dict())

    return wrapped


# Replace both decoder and discriminator mappings
new_genre_id = 16  # current max is 15
# Decoder
model.DanceDecoder.mapping = wrap_mapping_with_lora(
    mappingnet=model.DanceDecoder.mapping,
    in_dim=256,  # decoder
    dim=512,
    new_genre_id=16,
    lora_cfg=lora_cfg,
)

# Discriminator
model.dis_model.mapping = wrap_mapping_with_lora(
    mappingnet=model.dis_model.mapping,
    in_dim=512,  # discriminator input
    dim=1,
    new_genre_id=16,
    lora_cfg=lora_cfg,
)

# Bypass EMA
model.diffusion.master_model = model.diffusion.model
model.diffusion.master_model_dis = model.dis_model

# Freeze all except LoRA
freeze_all_except_lora(model.DanceDecoder)
freeze_all_except_lora(model.dis_model)
model

Local_Module(
  (DanceDecoder): Refine_DanceDecoder(
    (mapping): LoRA_MappingNet(
      (shared): Sequential(
        (0): Linear(in_features=256, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=512, out_features=512, bias=True)
        (3): GELU(approximate='none')
      )
      (unshared): ModuleDict(
        (0): Sequential(
          (0): Linear(in_features=512, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=512, bias=True)
        )
        (1): Sequential(
          (0): Linear(in_features=512, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=512, bias=True)
        )
        (2): Sequential(
          (0): Linear(in_features=512, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=512, bias=True)
        )
        

In [10]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

DanceDecoder.mapping.unshared.16.0.lora_A.lora.weight torch.Size([8, 512])
DanceDecoder.mapping.unshared.16.0.lora_B.lora.weight torch.Size([512, 8])
DanceDecoder.mapping.unshared.16.2.lora_A.lora.weight torch.Size([8, 512])
DanceDecoder.mapping.unshared.16.2.lora_B.lora.weight torch.Size([512, 8])
dis_model.mapping.unshared.16.0.lora_A.lora.weight torch.Size([8, 1])
dis_model.mapping.unshared.16.0.lora_B.lora.weight torch.Size([1, 8])
dis_model.mapping.unshared.16.2.lora_A.lora.weight torch.Size([8, 1])
dis_model.mapping.unshared.16.2.lora_B.lora.weight torch.Size([1, 8])


In [11]:
def test_lora_forward(mappingnet, genre_id, in_dim):
    x = torch.randn(2, in_dim)
    genre = torch.tensor([genre_id, genre_id])
    out = mappingnet(x, genre)
    print(f"✅ Output shape: {out.shape}")
    
test_lora_forward(model.DanceDecoder.mapping, genre_id=16, in_dim=256)
test_lora_forward(model.dis_model.mapping, genre_id=16, in_dim=512)

✅ Output shape: torch.Size([2, 512])
✅ Output shape: torch.Size([2, 1])


In [12]:
import copy

torch.save(model.DanceDecoder.mapping.state_dict(), "decoder_mapping_lora.pth")
mapping_copy = copy.deepcopy(model.DanceDecoder.mapping)
mapping_copy.load_state_dict(torch.load("decoder_mapping_lora.pth"))
print("✅ State dict loaded correctly")

✅ State dict loaded correctly


In [13]:
from peft.tuners.lora import Linear as LoraLinear

def patch_attention_with_lora(attn_module, lora_cfg, name_prefix=""):
    """Replace q_proj and v_proj inside a MultiheadAttention block with LoRA versions."""
    embed_dim = attn_module.embed_dim

    # Wrap original weights
    q_proj = LoraLinear(
        base_layer=nn.Linear(embed_dim, embed_dim, bias=False),
        adapter_name=f"{name_prefix}_q_lora",
        r=lora_cfg.r,
        lora_alpha=lora_cfg.lora_alpha,
        lora_dropout=lora_cfg.lora_dropout,
    )
    v_proj = LoraLinear(
        base_layer=nn.Linear(embed_dim, embed_dim, bias=False),
        adapter_name=f"{name_prefix}_v_lora",
        r=lora_cfg.r,
        lora_alpha=lora_cfg.lora_alpha,
        lora_dropout=lora_cfg.lora_dropout,
    )

    # Load pretrained weights
    q_proj.base_layer.weight.data.copy_(attn_module.in_proj_weight[:embed_dim])
    v_proj.base_layer.weight.data.copy_(attn_module.in_proj_weight[2*embed_dim:])

    # Monkey patch (store the remaining proj parts manually or skip them if not needed)
    attn_module.q_proj = q_proj
    attn_module.v_proj = v_proj

    # Overwrite forward pass (if necessary), or replace the module entirely
    return attn_module


In [14]:
def patch_transformer_with_lora(model, lora_cfg):
    # Patch encoder
    if hasattr(model, "cond_encoder"):
        for i, encoder_layer in enumerate(model.cond_encoder):
            patch_attention_with_lora(
                encoder_layer.self_attn,
                lora_cfg,
                name_prefix=f"encoder_layer_{i}"
            )

    # Patch decoder
    if hasattr(model, "seqTransDecoder"):
        for i, decoder_layer in enumerate(model.seqTransDecoder.stack):
            patch_attention_with_lora(
                decoder_layer.self_attn,
                lora_cfg,
                name_prefix=f"decoder_layer_{i}_self"
            )
            patch_attention_with_lora(
                decoder_layer.multihead_attn,
                lora_cfg,
                name_prefix=f"decoder_layer_{i}_cross"
            )


In [15]:
# Patch attention modules
patch_transformer_with_lora(model.DanceDecoder, lora_cfg)

# Optionally: freeze all but LoRA
freeze_all_except_lora(model.DanceDecoder)


In [21]:
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters:", sum(p.numel() for p in model.parameters()))

Trainable parameters: 409632
Total parameters: 73850689


In [17]:
from peft.tuners.lora import Linear as LoraLinear

def patch_discriminator_tr_block_with_lora(dis_model, lora_cfg):
    for i, block in enumerate(dis_model.tr_block.layers):
        ln1, attn, ln2, ff = block  # unpack 4 submodules in the ModuleList

        # Patch to_q
        lora_q = LoraLinear(
            base_layer=nn.Linear(attn.to_q.in_features, attn.to_q.out_features, bias=False),
            adapter_name=f"dis_tr_block_{i}_q",
            r=lora_cfg.r,
            lora_alpha=lora_cfg.lora_alpha,
            lora_dropout=lora_cfg.lora_dropout,
        )
        lora_q.base_layer.weight.data.copy_(attn.to_q.weight.data)
        attn.to_q = lora_q

        # Patch to_v
        lora_v = LoraLinear(
            base_layer=nn.Linear(attn.to_v.in_features, attn.to_v.out_features, bias=False),
            adapter_name=f"dis_tr_block_{i}_v",
            r=lora_cfg.r,
            lora_alpha=lora_cfg.lora_alpha,
            lora_dropout=lora_cfg.lora_dropout,
        )
        lora_v.base_layer.weight.data.copy_(attn.to_v.weight.data)
        attn.to_v = lora_v


In [18]:
from peft.tuners.lora import Linear as LoraLinear

def patch_discriminator_tr_block_with_lora(dis_model, lora_cfg):
    for i, block in enumerate(dis_model.tr_block.layers):
        ln1, attn, ln2, ff = block  # unpack 4 submodules in the ModuleList

        # Patch to_q
        lora_q = LoraLinear(
            base_layer=nn.Linear(attn.to_q.in_features, attn.to_q.out_features, bias=False),
            adapter_name=f"dis_tr_block_{i}_q",
            r=lora_cfg.r,
            lora_alpha=lora_cfg.lora_alpha,
            lora_dropout=lora_cfg.lora_dropout,
        )
        lora_q.base_layer.weight.data.copy_(attn.to_q.weight.data)
        attn.to_q = lora_q

        # Patch to_v
        lora_v = LoraLinear(
            base_layer=nn.Linear(attn.to_v.in_features, attn.to_v.out_features, bias=False),
            adapter_name=f"dis_tr_block_{i}_v",
            r=lora_cfg.r,
            lora_alpha=lora_cfg.lora_alpha,
            lora_dropout=lora_cfg.lora_dropout,
        )
        lora_v.base_layer.weight.data.copy_(attn.to_v.weight.data)
        attn.to_v = lora_v


In [19]:
patch_discriminator_tr_block_with_lora(model.dis_model, lora_cfg)
freeze_all_except_lora(model.dis_model)


In [20]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

DanceDecoder.mapping.unshared.16.0.lora_A.lora.weight torch.Size([8, 512])
DanceDecoder.mapping.unshared.16.0.lora_B.lora.weight torch.Size([512, 8])
DanceDecoder.mapping.unshared.16.2.lora_A.lora.weight torch.Size([8, 512])
DanceDecoder.mapping.unshared.16.2.lora_B.lora.weight torch.Size([512, 8])
DanceDecoder.cond_encoder.0.self_attn.q_proj.lora_A.encoder_layer_0_q_lora.weight torch.Size([8, 512])
DanceDecoder.cond_encoder.0.self_attn.q_proj.lora_B.encoder_layer_0_q_lora.weight torch.Size([512, 8])
DanceDecoder.cond_encoder.0.self_attn.v_proj.lora_A.encoder_layer_0_v_lora.weight torch.Size([8, 512])
DanceDecoder.cond_encoder.0.self_attn.v_proj.lora_B.encoder_layer_0_v_lora.weight torch.Size([512, 8])
DanceDecoder.cond_encoder.1.self_attn.q_proj.lora_A.encoder_layer_1_q_lora.weight torch.Size([8, 512])
DanceDecoder.cond_encoder.1.self_attn.q_proj.lora_B.encoder_layer_1_q_lora.weight torch.Size([512, 8])
DanceDecoder.cond_encoder.1.self_attn.v_proj.lora_A.encoder_layer_1_v_lora.weight 

In [ ]:
def extract_trainable_weights(model):
    """Extract trainable weights from the model."""
    trainable_weights = {}
    for name, param in model.named_parameters():
        if param.requires_grad:
            trainable_weights[name] = param.data.clone()
    return trainable_weights

# Usage
lora_weights = extract_trainable_weights(model)
# Save to file
torch.save(lora_weights, "lora_weights.pth")


{}

In [ ]:
# test load
lora_weights = torch.load("lora_weights.pth")

# Load the weights into the model
model.load_state_dict(lora_weights, strict=False)